In [1]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.iolib.summary2 import summary_col
from tabulate import tabulate
import numpy as np

In [2]:
train_panel_in = "../../../data/final/train_panel.parquet"
bike_panel_in = "../../../data/final/bike_panel.parquet"
uber_panel_in = "../../../data/final/uber_panel.parquet"

did_out = "../../../reports/replication/did_simple.md"

In [3]:
train_panel = pd.read_parquet(train_panel_in)
bike_panel = pd.read_parquet(bike_panel_in)
uber_panel = pd.read_parquet(uber_panel_in)

# Preprocessing

In [4]:
dotw_names = {f"C(dotw)[T.{i}]":j for i,j in zip([1,2,3,4],['Tuesday','Wednesday','Thursday','Friday'])}

In [5]:
# The DNC was only 4 days, not even friday. so weekends are uninformative.
# TODO: don't hard-code because we want to include more days for robustness check
train_panel = train_panel[(0 <= train_panel['dotw']) & (train_panel['dotw'] <= 3)]
bike_panel = bike_panel[(0 <= bike_panel['dotw']) & (bike_panel['dotw'] <= 3)]
uber_panel = uber_panel[(0 <= uber_panel['dotw']) & (uber_panel['dotw'] <= 3)]

# DiffNDiff

In [30]:
def mask_colinear(df, groupby, coef):
    test = df.groupby(groupby)[coef].nunique() > 1
    mask = df[groupby].isin(test[test].index)
    if not all(test):
        print(f"Dropping {sum(~test)} {groupby}, {sum(~mask)} obs invariant to {coef}")
    return mask

def did_colinearity(df):
    mask = np.full_like(df.index, True)
    mask &= mask_colinear(df, 'id', 'DNC')
    mask &= mask_colinear(df, 'date', 'UCMP')
    df = df.loc[mask]
    return df

def model_transit_did(df):
    df = df.sort_values('date')
    unit_obs = df.groupby('id').size()
    df['weight'] = df['id'].map(unit_obs).apply(lambda x: 1/x)
    # Assert that spatial units have time variance (only need to test group inclusion, not every variable.)
    df = did_colinearity(df)

    formula = f"""logrides ~ UCMP : DNC + id + C(time)"""
    did_model = sm.WLS.from_formula(formula, df, weights=df['weight']).fit(cov_type='cluster',cov_kwds={'groups': df['id']})
    return did_model

In [31]:
bike_did_model = model_transit_did(bike_panel)
bike_did_model.summary()

Dropping 458 id, 2840 obs invariant to DNC


/Users/eric/Repos/DNCTransit/.venv/lib/python3.10/site-packages/numpy/linalg/_linalg.py:3383: RuntimeWarning: divide by zero encountered in matmul
  return _core_matmul(x1, x2)
/Users/eric/Repos/DNCTransit/.venv/lib/python3.10/site-packages/numpy/linalg/_linalg.py:3383: RuntimeWarning: overflow encountered in matmul
  return _core_matmul(x1, x2)
/Users/eric/Repos/DNCTransit/.venv/lib/python3.10/site-packages/numpy/linalg/_linalg.py:3383: RuntimeWarning: invalid value encountered in matmul
  return _core_matmul(x1, x2)
/Users/eric/Repos/DNCTransit/.venv/lib/python3.10/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 1113, but rank is 56
  warnings.warn('covariance of constraints does not have full '


<class 'statsmodels.iolib.summary.Summary'>
"""
                            WLS Regression Results                            
==============================================================================
Dep. Variable:               logrides   R-squared:                       0.952
Model:                            WLS   Adj. R-squared:                  0.951
Method:                 Least Squares   F-statistic:                 1.292e+04
Date:                Mon, 12 Oct 2026   Prob (F-statistic):               0.00
Time:                        12:25:09   Log-Likelihood:                -17697.
No. Observations:               43303   AIC:                         3.762e+04
Df Residuals:                   42189   BIC:                         4.729e+04
Df Model:                        1113                                         
Covariance Type:              cluster                                         
==========================================================================================================================================
                                                                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------------------------------------------
Intercept                                                                  1.5013      0.020     74.593      0.000       1.462       1.541
id[T.21st St & Pulaski Rd]                                                -0.7018      0.002   -374.974      0.000      -0.706      -0.698
id[T.63rd St Beach]                                                        0.6202      0.001   1084.823      0.000       0.619       0.621
id[T.900 W Harrison St]                                                    2.5674      0.001   4491.139      0.000       2.566       2.569
id[T.Aberdeen St & Jackson Blvd]                                           2.9931      0.004    808.240      0.000       2.986       3.000
id[T.Aberdeen St & Monroe St]                                              2.5628      0.004    692.029      0.000       2.556       2.570
id[T.Aberdeen St & Randolph St]                                            2.9359      0.001   5135.712      0.000       2.935       2.937
id[T.Ada St & Washington Blvd]                                             2.4485      0.004    661.162      0.000       2.441       2.456
id[T.Adler Planetarium]                                                    3.1575      0.001   5523.252      0.000       3.156       3.159
id[T.Albany Ave & 16th St]                                                -0.8940      0.004   -224.194      0.000      -0.902      -0.886
id[T.Albany Ave & 26th St]                                                -0.6185      0.002   -343.575      0.000      -0.622      -0.615
id[T.Albany Ave & Belmont Ave]                                             0.0717      0.001    110.829      0.000       0.070       0.073
id[T.Albany Ave & Bloomingdale Ave]                                        1.9698      0.001   3445.673      0.000       1.969       1.971
id[T.Albany Ave & Douglas Blvd]                                           -0.6582      0.003   -231.015      0.000      -0.664      -0.653
id[T.Albany Ave & Montrose Ave]                                            1.2182      0.001   2131.024      0.000       1.217       1.219
id[T.Altgeld Gardens]                                                     -0.8120      0.005   -158.699      0.000      -0.822      -0.802
id[T.Archer (Damen) Ave & 37th St]                                        -0.1249      0.001   -118.597      0.000      -0.127      -0.123
id[T.Archer Ave & 43rd St]                                                -0.8673      0.005   -189.937      0.000      -0.876      -0.858
id[T.Archer Ave & 49th St]                                                -0.7661      0.005   -160.137      0.000      -0.775      -0.757
id[T.Archer Ave & Homan A

In [32]:
train_did_model = model_transit_did(train_panel)
train_did_model.summary()

/Users/eric/Repos/DNCTransit/.venv/lib/python3.10/site-packages/numpy/linalg/_linalg.py:3383: RuntimeWarning: divide by zero encountered in matmul
  return _core_matmul(x1, x2)
/Users/eric/Repos/DNCTransit/.venv/lib/python3.10/site-packages/numpy/linalg/_linalg.py:3383: RuntimeWarning: overflow encountered in matmul
  return _core_matmul(x1, x2)
/Users/eric/Repos/DNCTransit/.venv/lib/python3.10/site-packages/numpy/linalg/_linalg.py:3383: RuntimeWarning: invalid value encountered in matmul
  return _core_matmul(x1, x2)


<class 'statsmodels.iolib.summary.Summary'>
"""
                            WLS Regression Results                            
==============================================================================
Dep. Variable:               logrides   R-squared:                       0.025
Model:                            WLS   Adj. R-squared:                  0.018
Method:                 Least Squares   F-statistic:                     165.1
Date:                Mon, 12 Oct 2026   Prob (F-statistic):          9.25e-106
Time:                        12:25:13   Log-Likelihood:                -11061.
No. Observations:                7932   AIC:                         2.224e+04
Df Residuals:                    7874   BIC:                         2.264e+04
Df Model:                          57                                         
Covariance Type:              cluster                                         
==================================================================================
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept          8.7289      8.188      1.066      0.286      -7.319      24.776
C(time)[T.148]     0.7404      0.025     29.313      0.000       0.691       0.790
C(time)[T.149]     0.8102      0.026     31.078      0.000       0.759       0.861
C(time)[T.150]     0.8051      0.026     31.128      0.000       0.754       0.856
C(time)[T.154]     0.6995      0.024     29.406      0.000       0.653       0.746
C(time)[T.155]     0.8320      0.027     31.002      0.000       0.779       0.885
C(time)[T.156]     0.8417      0.027     31.677      0.000       0.790       0.894
C(time)[T.157]     0.8322      0.024     34.208      0.000       0.785       0.880
C(time)[T.161]     0.6520      0.025     26.407      0.000       0.604       0.700
C(time)[T.162]     0.7762      0.028     27.605      0.000       0.721       0.831
C(time)[T.163]     0.7955      0.026     30.351      0.000       0.744       0.847
C(time)[T.164]     0.7606      0.026     29.394      0.000       0.710       0.811
C(time)[T.168]     0.6279      0.025     25.020      0.000       0.579       0.677
C(time)[T.169]     0.7830      0.027     28.676      0.000       0.729       0.836
C(time)[T.170]     0.5634      0.021     26.209      0.000       0.521       0.606
C(time)[T.171]     0.7591      0.024     31.150      0.000       0.711       0.807
C(time)[T.175]     0.6561      0.022     29.748      0.000       0.613       0.699
C(time)[T.176]     0.7157      0.026     27.880      0.000       0.665       0.766
C(time)[T.177]     0.7999      0.025     32.555      0.000       0.752       0.848
C(time)[T.178]     0.8167      0.024     34.277      0.000       0.770       0.863
C(time)[T.182]     0.5687      0.082      6.975      0.000       0.409       0.729
C(time)[T.183]     0.6798      0.084      8.102      0.000       0.515       0.844
C(time)[T.184]     0.6632      0.082      8.045      0.000       0.502       0.825
C(time)[T.185]     0.0404      0.076      0.534      0.594      -0.108       0.189
C(time)[T.189]     0.5716      0.081      7.015      0.000       0.412       0.731
C(time)[T.190]     0.6537      0.084      7.774      0.000       0.489       0.818
C(time)[T.191]     0.6826      0.084      8.153      0.000       0.519       0.847
C(time)[T.192]     0.7020      0.084      8.342      0.000       0.537       0.867
C(time)[T.196]     0.5486      0.084      6.553      0.000       0.385       0.713
C(time)[T.197]     0.6550      0.086      7.628      0.000       0.487       0.823
C(time)[T.198]     0.6982      0.084      8.279      0.000       0.533       0.864
C(time)[T.199]     0.7067      0.084      8.425      0.000       0.542       0.871
C(time)[T.203]     0.6155      0.083      7.453      0.000       0.454       0.777
C(time)[T.204]     0.7138      0.085      8.408      0.000       0.547       0.880
C(time)[T.20

In [33]:
uber_did_model = model_transit_did(uber_panel)
uber_did_model.summary()

Dropping 130 id, 762 obs invariant to DNC


/Users/eric/Repos/DNCTransit/.venv/lib/python3.10/site-packages/numpy/linalg/_linalg.py:3383: RuntimeWarning: divide by zero encountered in matmul
  return _core_matmul(x1, x2)
/Users/eric/Repos/DNCTransit/.venv/lib/python3.10/site-packages/numpy/linalg/_linalg.py:3383: RuntimeWarning: overflow encountered in matmul
  return _core_matmul(x1, x2)
/Users/eric/Repos/DNCTransit/.venv/lib/python3.10/site-packages/numpy/linalg/_linalg.py:3383: RuntimeWarning: invalid value encountered in matmul
  return _core_matmul(x1, x2)


<class 'statsmodels.iolib.summary.Summary'>
"""
                            WLS Regression Results                            
==============================================================================
Dep. Variable:               logrides   R-squared:                       0.231
Model:                            WLS   Adj. R-squared:                  0.231
Method:                 Least Squares   F-statistic:                     22.14
Date:                Mon, 12 Oct 2026   Prob (F-statistic):          8.10e-144
Time:                        12:25:14   Log-Likelihood:            -1.0248e+05
No. Observations:               54949   AIC:                         2.051e+05
Df Residuals:                   54892   BIC:                         2.056e+05
Df Model:                          56                                         
Covariance Type:              cluster                                         
==================================================================================
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept       4.651e+04   2605.702     17.848      0.000    4.14e+04    5.16e+04
C(time)[T.148]     0.2185      0.048      4.556      0.000       0.124       0.312
C(time)[T.149]     0.4332      0.046      9.485      0.000       0.344       0.523
C(time)[T.150]     0.5433      0.049     11.108      0.000       0.447       0.639
C(time)[T.154]     0.1968      0.054      3.650      0.000       0.091       0.302
C(time)[T.155]     0.3903      0.047      8.381      0.000       0.299       0.482
C(time)[T.156]     0.4691      0.051      9.283      0.000       0.370       0.568
C(time)[T.157]     0.4426      0.058      7.668      0.000       0.329       0.556
C(time)[T.161]     0.1562      0.048      3.277      0.001       0.063       0.250
C(time)[T.162]     0.2254      0.048      4.712      0.000       0.132       0.319
C(time)[T.163]     0.3162      0.047      6.773      0.000       0.225       0.408
C(time)[T.164]     0.4537      0.049      9.297      0.000       0.358       0.549
C(time)[T.168]     0.1788      0.046      3.911      0.000       0.089       0.268
C(time)[T.169]     0.3229      0.046      6.954      0.000       0.232       0.414
C(time)[T.170]     0.3112      0.058      5.344      0.000       0.197       0.425
C(time)[T.171]     0.4171      0.049      8.543      0.000       0.321       0.513
C(time)[T.175]     0.1419      0.057      2.495      0.013       0.030       0.253
C(time)[T.176]     0.3496      0.048      7.324      0.000       0.256       0.443
C(time)[T.177]     0.3343      0.058      5.748      0.000       0.220       0.448
C(time)[T.178]     0.3764      0.052      7.179      0.000       0.274       0.479
C(time)[T.182]     0.2109      0.045      4.714      0.000       0.123       0.299
C(time)[T.183]     0.3145      0.047      6.636      0.000       0.222       0.407
C(time)[T.184]     0.4557      0.055      8.316      0.000       0.348       0.563
C(time)[T.185]     0.3409      0.059      5.767      0.000       0.225       0.457
C(time)[T.189]     0.1359      0.042      3.274      0.001       0.055       0.217
C(time)[T.190]     0.3474      0.045      7.656      0.000       0.258       0.436
C(time)[T.191]     0.3328      0.061      5.472      0.000       0.214       0.452
C(time)[T.192]     0.4229      0.062      6.855      0.000       0.302       0.544
C(time)[T.196]     0.2278      0.054      4.202      0.000       0.122       0.334
C(time)[T.197]     0.2987      0.051      5.870      0.000       0.199       0.398
C(time)[T.198]     0.3146      0.051      6.122      0.000       0.214       0.415
C(time)[T.199]     0.4281      0.052      8.195      0.000       0.326       0.530
C(time)[T.203]     0.1727      0.056      3.063      0.002       0.062       0.283
C(time)[T.204]     0.2964      0.049      6.058      0.000       0.200       0.392
C(time)[T.20

In [39]:
headers = ["DiD (Uber)", "DiD (Train)", "DiD (Bike)"]

did_summary = summary_col([uber_did_model, train_did_model, bike_did_model], 
            model_names=headers, 
            info_dict={"N":lambda x:(x.nobs)},
            stars=True,
            regressor_order=["UCMP:DNC", "id", "C(time)"],
            drop_omitted=True)

did_summary.tables[0].index = (did_summary.tables[0].index
                               .str.replace('DNC','During DNC')
                               .str.replace('UCMP','Near DNC')
                               .str.replace('C(time)','time fe')
                               .str.replace('id','unit fe'))

did_table = tabulate(did_summary.tables[0],
                   headers=[""] + headers,
                   showindex=True,
                   tablefmt='github')
print(did_table)
with open(did_out,"w") as f:
    f.write(did_table)

# !cp {did_out} ../../../../eric-mc2-cv/static/uploads/

|                     | DiD (Uber)   | DiD (Train)   | DiD (Bike)   |
|---------------------|--------------|---------------|--------------|
| Near DNC:During DNC | 2.2260***    | 0.0231        | 0.2593***    |
|                     | (0.3363)     | (0.2450)      | (0.0510)     |
| unit fe             | -0.0000***   | -0.0000       |              |
|                     | (0.0000)     | (0.0002)      |              |
| R-squared           | 0.2314       | 0.0246        | 0.9521       |
| R-squared Adj.      | 0.2306       | 0.0175        | 0.9508       |
| N                   | 54949.0      | 7932.0        | 43303.0      |


## DiD Impact

In [35]:
beta3 = did_summary.tables[0].xs('Near DNC:During DNC').str.replace('*','').astype(float)
main_pct = np.exp(beta3) - 1
pct3 = main_pct.apply("{:.1%}".format)
pct3

DiD (Uber)     826.3%
DiD (Train)      2.3%
DiD (Bike)      29.6%
Name: Near DNC:During DNC, dtype: object